# 04 — Model Improvement and Analysis

## 4.1 Objectives

This notebook investigates weaknesses in the existing ResNet18 baseline and evaluates a targeted inference improvement.

The objectives are to:

- Identify weaker classes using the saved validation metrics.
- Evaluate horizontal-flip test-time augmentation (TTA).
- Compare both inference methods on identical validation images.
- Determine whether the proposed change improves accuracy on the experimental subset.
- Distinguish tested changes from future improvement strategies.


## 4.2 Setup and baseline metrics

In [7]:
from pathlib import Path
import json
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"
ARTIFACTS = ROOT / "artifacts" / "baseline"
sys.path.insert(0, str(ROOT / "src"))

from enge707.data import FacialFeatureDataset, build_transforms
from enge707.model import build_resnet18

config = json.loads((ARTIFACTS / "config.json").read_text())
metrics = json.loads((ARTIFACTS / "metrics.json").read_text())
validation = pd.read_csv(ARTIFACTS / "validation_split.csv", dtype={"image_id": str})

classes = config["classes"]
class_to_index = config["class_to_index"]

print("Baseline accuracy:", f"{metrics['validation_accuracy']:.2%}")

Baseline accuracy: 75.35%


## 4.3 Identify weaker classes

In [8]:
report = pd.DataFrame(metrics["classification_report"]).T

class_metrics = report.loc[classes, ["precision", "recall", "f1-score", "support"]]

display(class_metrics.sort_values("recall").round(3))

,precision,recall,f1-score,support
CatMouth,0.620,0.219,0.324,1388.0
DogMouth,0.702,0.351,0.468,1158.0
DogNose,0.538,0.752,0.628,1158.0
CatNose,0.513,0.799,0.625,1388.0
DogEye,0.840,0.843,0.842,2316.0
CatFace,0.991,0.890,0.938,1389.0
DogFace,0.968,0.890,0.928,1158.0
CatEye,0.827,0.966,0.891,2777.0


## 4.4 Controlled TTA experiment

In [9]:
checkpoint = torch.load(
    ARTIFACTS / "best_model.pt",
    map_location="cpu",
    weights_only=False,
)

assert checkpoint["classes"] == classes

model = build_resnet18(
    num_classes=len(classes),
    pretrained=False,
)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Saved model loaded successfully.")

Saved model loaded successfully.


In [ ]:
subset = (
    validation.groupby("class")
    .sample(n=16, random_state=42)
    .sort_index()
    .reset_index(drop=True)
)

_, evaluation_transform = build_transforms()

dataset = FacialFeatureDataset(
    subset,
    DATA / "train" / "train",
    evaluation_transform,
    class_to_index,
)

loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)

print("Experimental images:", len(dataset))

Experimental images: 128


## 4.5 Compare normal inference against TTA

In [ ]:
normal_correct = 0
tta_correct = 0
total = 0

with torch.inference_mode():
    for images, labels in loader:
        normal_logits = model(images)

        flipped = torch.flip(images, dims=[3])
        flipped_logits = model(flipped)

        tta_logits = (normal_logits + flipped_logits) / 2

        normal_correct += (normal_logits.argmax(dim=1) == labels).sum().item()

        tta_correct += (tta_logits.argmax(dim=1) == labels).sum().item()

        total += len(labels)

comparison = pd.DataFrame(
    [
        {
            "Method": "Normal inference",
            "Correct": normal_correct,
            "Total": total,
            "Accuracy": normal_correct / total,
        },
        {
            "Method": "Horizontal-flip TTA",
            "Correct": tta_correct,
            "Total": total,
            "Accuracy": tta_correct / total,
        },
    ]
)

display(comparison.style.format({"Accuracy": "{:.2%}"}))

difference = 100 * (tta_correct - normal_correct) / total
print(f"TTA accuracy difference: {difference:+.2f} percentage points")

,Method,Correct,Total,Accuracy
0,Normal inference,93,128,72.66%
1,Horizontal-flip TTA,93,128,72.66%


TTA accuracy difference: +0.00 percentage points



## 4.6 Findings and Limitations

Horizontal-flip test-time augmentation was evaluated using 128 validation images, with 16 images sampled reproducibly from each class.

The experiment compared standard inference and horizontally flipped ensemble predictions using the same saved ResNet18 checkpoint.

The measured difference is reported above. This experiment does not establish an improvement across the entire validation dataset or Kaggle test set.

**Limitations:**

- The subset is relatively small, so its accuracy estimates are uncertain.
- TTA increases computational cost by requiring two forward passes.
- Horizontal flipping may not benefit every class.
- No model parameters were updated during this experiment.

Potential future strategies include additional training epochs, class-weighted loss, improved low-resolution image handling and hyperparameter tuning. These are proposals, not completed experiments.
